<a href="https://colab.research.google.com/github/3702880330101-netizen/Smart-Cost/blob/main/SmartCost_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SmartCost AI

## Gemelo economico para experimentar antes de decidir

Prototipo de la **Feria de Innovacion 2026** para la carrera de **Ingenieria de Costos**.

Este notebook es el **motor de calculo** del proyecto. La interfaz web de GitHub Pages no
calcula nada: solo pide datos y dibuja lo que este notebook responde.

### Las tres reglas que sostienen todo el proyecto

1. **Las nueve formulas del documento son el unico calculo.** Viven programadas en Python,
   en la carpeta `colab/`, y se pueden leer linea por linea.
2. **La IA interpreta, no calcula.** Recibe un JSON con resultados ya calculados y devuelve
   texto: diagnostico, variables criticas, acciones y riesgos. Si falta un dato, lo dice.
3. **La hoja de control manda.** Cada prueba compara el motor contra una hoja de calculo
   independiente, con tolerancia de 0.000001.

### Que vas a ver en esta sesion

| # | Seccion |
|---|---------|
| 1 | Presentacion y alcance |
| 2 | Preparacion del entorno |
| 3 | Carga de los modulos del motor |
| 4 | El contrato: las nueve variables de entrada |
| 5 | Hoja de control: las nueve formulas |
| 6 | PRUEBA 1: escenario base |
| 7 | PRUEBAS 2, 3 y 4: variaciones y caso limite |
| 8 | Escenarios optimista y adverso |
| 9 | Analisis de sensibilidad |
| 10 | Interpretacion con IA (Gemini) y auditoria |
| 11 | Bitacora de uso de la IA |
| 12 | Levantar la API para la interfaz web |
| 13 | Conclusiones, supuestos y limitaciones |

---

Si algo no esta claro o el modelo dice algo que no se puede explicar con una formula, no se
acepta. Un resultado que el equipo no puede defender delante de un jurado no sirve.

## 1. Que resuelve este prototipo

Un problema concreto de Ingenieria de Costos: **responder la pregunta de si conviene cambiar
algo, antes de cambiarlo.**

Un estudiante o un analista de planta tiene un costo y una utilidad. Cambiar el precio del
material, mejorar un proceso o subir el volumen "suena" a mejora, pero casi nunca lo es en la
misma medida. El riesgo es mover la variable equivocada y empeorar el resultado sin notarlo.

SmartCost AI es un **gemelo economico**: una copia ejecutable del calculo de costos de una
sola operacion. En lugar de cambiar algo y ver que pasa, se pregunta que pasaria si se
cambiara, y se mide con las mismas nueve formulas del curso.

Lo que hace:

- Calcula costo, margen, punto de equilibrio y utilidad de un escenario de nueve variables.
- Genera cuatro escenarios y compara cuanto cambia la utilidad en cada uno.
- Mide cuanto impacta cada variable, moviendola una por vez y ordenando el resultado.
- Entrega esos numeros ya calculados a una IA, que los interpreta en lenguaje de negocio.
- Comprueba sus propios resultados contra una hoja de calculo independiente.

Lo que **no** hace, y por principio es importante decirlo antes que despues: no predice la
demanda, no conoce la capacidad instalada, no modela inventario ni impuestos, y no
recomienda nada sin que exista un numero detras.

## 2. Preparacion del entorno

Ejecuta esta celda una sola vez por sesion de Colab. Instala el servidor web y el cliente de
Gemini. La clave de la IA **no** se instala aqui: esa va en un secreto de Colab, en la seccion 10.

Si la sesion ya tiene estas librerias, la celda no hace nada.

Los paquetes se instalan por nombre y no desde el archivo para que esta celda tambien
funcione cuando solo se subieron los `.py` de `colab/`. Si clonaste el repositorio
completo, `pip install -r requirements.txt` instala exactamente lo mismo.

In [1]:
!pip -q install flask flask-cors google-genai pyngrok
print('Dependencias listas.')

Dependencias listas.


## 3. Carga de los modulos del motor

El motor de calculo son los archivos de la carpeta `colab/` del repositorio. Esta celda los
trae a Colab. Elige **una** de las dos formas:

**Forma A - desde el repositorio (si es publico).** Pega la URL base en `REPO_RAW` y ejecuta.

**Forma B - subida manual.** Si el repositorio es privado o no hay internet, deja `REPO_RAW`
vacio: se abre el selector de archivos de Colab y subes los ocho `.py` de la carpeta `colab/`.

Las formulas **no** estan escritas en este notebook. Viven en los modulos, para que exista una
sola copia de la logica de negocio en todo el proyecto y se pueda auditar.

In [ ]:
REPO_RAW = ''  # ejemplo: 'https://raw.githubusercontent.com/USUARIO/REPO/main'

import os, sys, glob, urllib.request

RUTA_MODULOS = 'colab'
NOMBRES = ['config', 'motor', 'escenarios', 'sensibilidad',
           'ia_gemini', 'hoja_control', 'pruebas', 'api_server']

def _descargar(repo_raw):
    os.makedirs(RUTA_MODULOS, exist_ok=True)
    for nombre in NOMBRES:
        url = repo_raw.rstrip('/') + '/' + RUTA_MODULOS + '/' + nombre + '.py'
        destino = os.path.join(RUTA_MODULOS, nombre + '.py')
        urllib.request.urlretrieve(url, destino)
    print('Modulos descargados en ' + RUTA_MODULOS + '/')

def _subir():
    from google.colab import files
    os.makedirs(RUTA_MODULOS, exist_ok=True)
    for nombre, contenido in files.upload().items():
        if nombre.endswith('.py'):
            destino = os.path.join(RUTA_MODULOS, os.path.basename(nombre))
            with open(destino, 'wb') as archivo:
                archivo.write(contenido)
    print('Modulos subidos en ' + RUTA_MODULOS + '/')

faltantes = [n for n in NOMBRES if not glob.glob(RUTA_MODULOS + '/' + n + '.py')]
if not faltantes:
    print('Los modulos ya estaban disponibles.')
elif REPO_RAW.strip():
    _descargar(REPO_RAW)
else:
    print('Faltan ' + str(len(faltantes)) + ' modulos. Se abre el selector de archivos:')
    print('sube los .py de la carpeta colab/ del repositorio.')
    _subir()

Faltan 8 modulos. Se abre el selector de archivos:
sube los .py de la carpeta colab/ del repositorio.


In [ ]:
if RUTA_MODULOS not in sys.path:
    sys.path.insert(0, RUTA_MODULOS)

import config, motor, escenarios, sensibilidad
import ia_gemini, hoja_control, pruebas, api_server

print(config.NOMBRE_PROYECTO, 'motor ' + config.VERSION_MOTOR)
print('Reglas: ' + config.VERSION_REGLAS)
print('Variables de entrada:', ', '.join(config.CLAVES_ENTRADA))
print('Escenarios:', ', '.join(config.NOMBRES_ESCENARIOS))

faltan = [n for n in NOMBRES if n not in sys.modules]
print('Faltan modulos por cargar: ' + (', '.join(faltan) if faltan else 'ninguno'))

## 4. El contrato: las nueve variables de entrada

Estas son las unicas nueve variables que el modelo acepta. Los nombres tecnicos de la
columna `clave` son exactamente los que viajan en el JSON hacia Python y los que aparecen en la
trazabilidad, y **no se renombran nunca**: esa consistencia es lo que permite auditar un
resultado sin adivinar de que dato habla.

Fíjate en `desperdicio`: se maneja en **porcentaje**. El dato `4` significa 4 por ciento. La
conversion a factor ocurre en un unico lugar del motor, `factor_desperdicio()`, precisamente para
que no haya dos caminos distintos de convertir el mismo dato.

In [ ]:
print('%-18s %-22s %-14s %s' % ('clave', 'unidad', 'rango', 'positivo'))
print('-' * 72)
for campo in config.ENTRADAS:
    rango = '%s a %s' % (motor.num(campo['min'], 2), motor.num(campo['max'], 2))
    print('%-18s %-22s %-14s %s' % (campo['clave'], campo['unidad'], rango, campo['positivo']))

print()
print('Caso base de la PRUEBA 1:')
for clave in config.CLAVES_ENTRADA:
    print('  %-18s %s' % (clave, motor.num(config.CASO_BASE[clave], 4)))

## 5. Hoja de control: las nueve formulas

El documento entrega la logica de calculo en formulas de hoja de calculo. Este proyecto las
tradujo a Python, una por una, sin agregar pasos intermedios.

La hoja de control se diseña asi: **columna A** el nombre del resultado, **columna B** el valor,
**columna C** la formula. Las entradas ocupan B2 a B10. Las formulas de la columna C se pueden
copiar tal cual en Excel o Google Sheets.

La celda siguiente imprime las nueve formulas del documento junto a la version programada, para
que se puedan comparar visualmente.

In [ ]:
for clave, formula in hoja_control.FORMULAS_HOJA.items():
    print(clave)
    print('   hoja de control : ' + formula)
    print('   en el motor     : ' + motor.FORMULAS.get(clave, '(derivada) ' + motor.FORMULAS_DERIVADAS.get(clave, '')))
print()
print('Tolerancia de comparacion: ' + str(hoja_control.TOLERANCIA))
print('Casos de prueba: ' + ', '.join(hoja_control.CASOS.keys()))

## 6. PRUEBA 1: escenario base

La primera prueba exigida por el documento. El motor corre las nueve formulas y el resultado se
compara contra la hoja de control, indicador por indicador.

In [ ]:
CASO_BASE = dict(config.CASO_BASE)

def comparar_con_hoja(entradas, caso):
    '''Corre el motor y compara contra la hoja. Devuelve el resultado y la peor desviacion.'''
    resultado = motor.calcular_escenario(entradas)
    esperado = hoja_control.CASOS[caso]['esperado']
    peor = 0.0
    for clave in hoja_control.INDICADORES_A_COMPARAR:
        valor_hoja = esperado.get(clave)
        valor_motor = resultado.get(clave)
        if valor_hoja is None and valor_motor is None:
            continue
        if valor_hoja is None or valor_motor is None:
            return resultado, float('inf')
        peor = max(peor, abs(float(valor_hoja) - float(valor_motor)))
    return resultado, peor

def mostrar_resultado(resultado):
    for clave in ['material_ajustado', 'costo_mp_unitario', 'costo_mod_unitario',
                  'cif_variable', 'costo_variable_unitario', 'costos_fijos',
                  'costo_unitario', 'margen_contribucion_unitario',
                  'margen_contribucion_porcentual', 'punto_equilibrio',
                  'utilidad', 'advertencias']:
        if clave not in resultado:
            continue
        valor = resultado[clave]
        if clave == 'advertencias':
            print('  advertencias:')
            for aviso in valor or []:
                print('     - ' + str(aviso.get('mensaje', aviso)))
        elif clave in ('costo_variable_unitario', 'costo_unitario',
                       'margen_contribucion_unitario', 'costos_fijos', 'utilidad'):
            print('  %-32s %s' % (clave, motor.q(valor)))
        elif clave in ('margen_contribucion_porcentual',):
            print('  %-32s %s' % (clave, motor.pct(valor)))
        else:
            print('  %-32s %s' % (clave, motor.num(valor, 4)))

resultado_base, desviacion = comparar_con_hoja(CASO_BASE, 'base')
mostrar_resultado(resultado_base)
print()
print('PRUEBA 1: escenario base')
print('  Desviacion maxima contra la hoja de control: ' + str(desviacion))
print('  Resultado: ' + ('APROBADA' if desviacion <= hoja_control.TOLERANCIA else 'FALLIDA'))

In [ ]:
print('Trazabilidad del escenario base: de donde sale cada numero')
print()
for fila in motor.trazabilidad(CASO_BASE, resultado_base):
    print(fila['resultado'])
    print('   formula  : ' + fila['formula'])
    print('   operandos: ' + fila['operandos'])
    print('   valor    : ' + fila['valor'])

## 7. PRUEBAS 2, 3 y 4: variaciones y caso limite

Las otras tres pruebas del documento.

- **PRUEBA 2:** el precio del material sube 12 por ciento.
- **PRUEBA 3:** el desperdicio pasa de 4 a 9 por ciento.
- **PRUEBA 4:** caso limite con margen no positivo. El motor debe avisar y **no** inventar un
  punto de equilibrio cuando el margen no alcanza para cubrir los costos fijos.

En los tres casos el cambio se hace sobre **una o dos variables**; todo lo demas lo resuelve el
motor.

In [ ]:
PRUEBAS = [
    ('precio_material_mas_12', 'PRUEBA 2: precio del material +12 %',
     {'precio_material': 20.16}),
    ('desperdicio_4_a_9', 'PRUEBA 3: desperdicio de 4 % a 9 %',
     {'desperdicio': 9.0}),
    ('extremo_margen_no_positivo', 'PRUEBA 4: caso limite con margen no positivo',
     {'precio_venta': 25.0}),
]

for caso, titulo, cambios in PRUEBAS:
    entradas = dict(CASO_BASE)
    entradas.update(cambios)
    resultado, desviacion = comparar_con_hoja(entradas, caso)
    print('=' * 70)
    print(titulo)
    for clave in sorted(cambios):
        print('   cambio: %s = %s' % (clave, motor.num(cambios[clave], 4)))
    print('   costo unitario   : ' + motor.q(resultado['costo_unitario']))
    print('   margen unitario  : ' + motor.q(resultado['margen_contribucion_unitario']))
    print('   margen porcentual: ' + motor.pct(resultado['margen_contribucion_porcentual']))
    if resultado['punto_equilibrio_valido']:
        print('   punto de equilibrio: ' + motor.num(resultado['punto_equilibrio'], 2) + ' unidades')
    else:
        print('   punto de equilibrio: NO CALCULABLE')
    print('   utilidad         : ' + motor.q(resultado['utilidad']))
    for aviso in resultado.get('advertencias') or []:
        print('   aviso            : ' + str(aviso.get('mensaje', aviso)))
    print('   desviacion maxima : ' + str(desviacion))
    print('   resultado        : ' + ('APROBADA' if desviacion <= hoja_control.TOLERANCIA else 'FALLIDA'))

## 8. Escenarios optimista y adverso

El motor construye cuatro escenarios con **la misma funcion**, `calcular_escenario()`:

- **Base:** los datos que se fijaron como referencia.
- **Nuevo:** lo que el usuario esta escribiendo ahora.
- **Optimista** y **Adverso:** la referencia con variaciones aplicadas.

Que se mueve en cada escenario no lo inventa la IA: son porcentajes declarados y visibles, que
el usuario puede editar desde la pagina web antes de simular.

Un detalle importante: **`material` nunca se mueve automaticamente.** El consumo estandar es una
decision de ingenieria, no un precio. Moverlo sin autorizacion seria inventar un dato de
produccion. Si se quiere cambiar, el usuario lo cambia como variable de entrada.

In [ ]:
paquete = escenarios.generar_escenarios(CASO_BASE)

print('Utilidad por escenario')
print('%-12s %16s %16s %14s' % ('escenario', 'costo unitario', 'utilidad', 'p. equilibrio'))
for nombre in config.NOMBRES_ESCENARIOS:
    resultado = paquete[nombre]['resultado']
    equilibrio = (motor.num(resultado['punto_equilibrio'], 2) + ' u'
                   if resultado['punto_equilibrio_valido'] else 'NO CALCULA')
    print('%-12s %16s %16s %14s' % (paquete[nombre]['etiqueta'],
          motor.q(resultado['costo_unitario']),
          motor.q(resultado['utilidad']), equilibrio))

print()
for nombre in ['optimista', 'adverso']:
    print('Que se movio en ' + paquete[nombre]['etiqueta'].lower() + ':')
    for cambio in paquete[nombre]['cambios']:
        print('   ' + cambio['texto'])
    print()

In [ ]:
tabla = escenarios.tabla_comparativa(paquete)
print('Tabla comparativa de indicadores')
print()
for indicador in tabla['indicadores']:
    print(indicador['etiqueta'])
    for nombre_esc in tabla['columnas']:
        celda = indicador['valores'][nombre_esc]
        texto = 'NO CALCULABLE' if celda['es_null'] else celda['texto']
        print('   %-12s %s' % (nombre_esc, texto))

print('Mejor escenario: ' + str(paquete.get('mejor_escenario', 'no definido')))
print('Peor escenario : ' + str(paquete.get('peor_escenario', 'no definido')))

## 9. Analisis de sensibilidad

Aqui es donde la IA suele equivocarse, asi que conviene ser estricto.

El motor toma cada variable por turno, la mueve un 10 por ciento hacia arriba, vuelve a correr
las nueve formulas, y luego un 10 por ciento hacia abajo. La diferencia de utilidad entre esas
corridas es el **impacto** de la variable. Al final ordena de mayor a menor.

Ese orden es un numero, no una opinion. Cuando el texto de la IA diga que una variable es
critica, se esta igualando a esta tabla, no a su intuicion. Si la IA inventa un orden distinto,
la auditoria lo detecta.

In [ ]:
filas = sensibilidad.analizar_sensibilidad(paquete['nuevo']['datos'])
criticas = sensibilidad.variables_criticas(filas)

print('Sensibilidad (paso de ' + motor.pct(100 * filas[0]['paso_relativo']) + ')')
print('%-4s %-18s %16s %12s %12s' % ('#', 'variable', 'impacto utilidad', '% cambio', 'favorable'))
for fila in filas:
    print('%-4d %-18s %16s %12s %12s' % (
        fila['posicion'], fila['etiqueta'], motor.q(fila['impacto']),
        motor.pct(fila['impacto_porcentual']), fila['direccion_favorable']))

print()
print('Las tres variables criticas, segun el motor:')
for critica in criticas:
    print('   %d. %-18s impacto %s (%s) favorable: %s' % (
        critica['posicion'], critica['etiqueta'], motor.q(critica['impacto_utilidad']),
        motor.pct(critica['impacto_porcentual']), critica['direccion_favorable']))

## 10. Interpretacion con IA y auditoria

### La clave, en un secreto de Colab

La clave de Gemini **nunca** se escribe en el codigo, ni en el notebook, ni en la pagina web.
Va en el panel de secretos de Colab:

1. Menu de la izquierda, icono de llave, **Secrets (secrets)**.
2. **Add a new secret**, nombre exactamente `GEMINI_API_KEY`, valor tu clave.
3. En **Notebook access** dale acceso al notebook.

### Que recibe el modelo

No recibe las nueve variables para que calcule. Recibe un JSON con los cuatro escenarios ya
calculados, la sensibilidad ya ordenada y los supuestos del modelo. El prompt le dice
explicitamente que no sustituya ninguna formula.

Si no hay clave, o la llamada falla, el motor genera **su propia** revision por reglas y la
marca como `motor` en vez de `gemini`. La pagina web lo dice. Codigo nunca se presenta como IA.

In [ ]:
variacion = escenarios.variacion_utilidad(paquete)

hay_clave = ia_gemini.hay_clave_api()
print('Hay clave de Gemini disponible: ' + ('si' if hay_clave else 'no'))
if not hay_clave:
    print()
    print('Sin clave, la celda siguiente igual va a funcionar: devolvera la revision automatica')
    print('del motor y la marcara como tal. Para usar la IA real, agrega el secreto')
    print('GEMINI_API_KEY en el panel de secretos de Colab y vuelve a ejecutar esta celda.')

analisis = ia_gemini.interpretar(paquete, filas, criticas, variacion)

print()
print('Fuente del texto: ' + analisis['etiqueta_fuente'])
if analisis.get('modelo'):
    print('Modelo: ' + str(analisis['modelo']))
if analisis.get('motivo_sin_ia'):
    print('Motivo: ' + analisis['motivo_sin_ia'])

print()
print('DIAGNOSTICO')
print(analisis['diagnostico'])

In [ ]:
for titulo, clave in [('VARIABLES CRITICAS', 'variables_criticas'),
                      ('ACCIONES A SIMULAR', 'acciones'),
                      ('RIESGOS Y SUPUESTOS', 'riesgos')]:
    print()
    print(titulo)
    for punto in analisis[clave]:
        print('   - ' + str(punto))

print()
print('AUDITORIA DE LA RESPUESTA')
auditoria = analisis['auditoria']
print('  ' + auditoria['resultado'])
print('  Metodo: ' + auditoria['metodo'])
if auditoria['advertencias']:
    print('  Correcciones aplicadas a la respuesta:')
    for advertencia in auditoria['advertencias']:
        print('     - ' + str(advertencia))
if analisis.get('secciones_faltantes'):
    print('  Secciones que faltaron: ' + ', '.join(analisis['secciones_faltantes']))

### Que se le entrega a la IA

Este bloque imprime el JSON exacto que recibe el modelo. Sirve para demostrar, delante del
jurado, que la IA recibio **resultados calculados** y no variables libres para que inventara
números.

In [ ]:
import json

envio = analisis['payload_enviado_a_la_ia']
print('Tamano del JSON entregado a la IA: ' + str(envio['tamano_caracteres']) + ' caracteres')
print()
print(json.dumps(envio['payload'], indent=2, ensure_ascii=False)[:4000])
print()
print('(La vista esta truncada a 4000 caracteres para no saturar la salida.)')

## 11. Bitacora de uso de la IA

El documento exige dejar registro de que hizo la IA, que hizo el codigo y que decidio una
persona. El modulo `ia_gemini` registra cada llamada automaticamente, con fecha, herramienta,
para que se uso, la salida y como se valido.

In [ ]:
registros = ia_gemini.bitacora_completa()
print('Llamadas registradas en esta sesion: ' + str(len(registros)))
print()
for registro in registros:
    print('Fecha      : ' + str(registro.get('fecha')))
    print('Herramienta: ' + str(registro.get('herramienta')))
    print('Uso        : ' + str(registro.get('uso')))
    print('Estado     : ' + str(registro.get('estado')))
    print('Validacion : ' + str(registro.get('validacion')))
    print('Secciones que faltaron: ' + (', '.join(registro.get('secciones_faltantes') or []) or 'ninguna'))
    print('Observaciones: ' + (str(registro.get('observaciones')) or 'ninguna'))
    print('-' * 70)

## 12. Levantar la API para la interfaz web

El motor se expone como una API HTTP. Esta es la parte que permite que la pagina de GitHub
Pages use estos calculos sin nada instalado en el navegador del usuario.

### Rutas disponibles

| Ruta | Metodo | Que hace |
|------|--------|----------|
| `/health` | GET | Comprueba que el motor vive y si hay clave de IA |
| `/contrato` | GET | Devuelve las nueve entradas, sus limites y los deltas por defecto |
| `/simulate` | POST | Corre la simulacion completa y devuelve el JSON que dibuja la pagina |
| `/pruebas` | GET | Ejecuta las cuatro pruebas obligatorias y devuelve el informe |

Ejecuta la celda siguiente. Imprime una URL publica: esa es la que se pega en la seccion 0 de
la pagina web. **La URL cambia en cada sesion**, porque el tunel es temporal.

In [ ]:
api_server.iniciar_servidor(config.PUERTO_SERVIDOR, usar_ngrok=True)

### Comprobar la API desde aqui mismo

Las celdas siguientes llaman al servidor recien levantado. Si funcionan, la pagina web tambien
va a funcionar.

In [ ]:
import urllib.request

def llamar(ruta, cuerpo=None):
    base = 'http://127.0.0.1:' + str(config.PUERTO_SERVIDOR)
    if cuerpo is None:
        peticion = urllib.request.Request(base + ruta)
    else:
        peticion = urllib.request.Request(
            base + ruta, data=json.dumps(cuerpo).encode('utf-8'),
            headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(peticion, timeout=60) as respuesta:
        return json.loads(respuesta.read().decode('utf-8'))

salud = llamar('/health')
print('GET /health   -> ' + str(salud.get('estado')) + ', IA disponible: ' + str(salud.get('ia_disponible')))

contrato = llamar('/contrato')
print('GET /contrato -> ' + str(len(contrato['claves_entrada'])) + ' variables de entrada')

respuesta = llamar('/simulate', CASO_BASE)
print('POST /simulate -> costo unitario ' + motor.q(respuesta['resultados']['costo_unitario'])
      + ', utilidad ' + motor.q(respuesta['resultados']['utilidad']))
print('  La clave que recibe la pagina es: resultados, escenarios, sensibilidad_detalle,')
print('  graficos, trazabilidad, metricas, hoja_control y el bloque ia.')

### Ejecutar las cuatro pruebas obligatorias

La celda siguiente corre la suite completa: las cuatro pruebas obligatorias del documento y seis
pruebas de control del propio prototipo.

| # | Prueba | Que protege |
|---|--------|-------------|
| 1 | Escenario base | Coincide con la hoja de control |
| 2 | Precio del material +12 % | Se propaga por toda la cadena |
| 3 | Desperdicio de 4 % a 9 % | Se refleja en el margen |
| 4 | Margen no positivo | El punto de equilibrio no existe y se advierte |
| 5 | Validaciones | Ninguna entrada invalida pasa en silencio |
| 6 | Escenarios del mismo motor | Los cuatro salen de la misma funcion |
| 7 | Sensibilidad ordenada | El orden sale del impacto medido |
| 8 | La IA no calcula | El bloque de IA no introduce numeros nuevos |
| 9 | Hoja de control completa | Los cuatro casos, indicador por indicador |
| 10 | Sin IA sigue funcionando | Responde igual sin clave de Gemini |

In [ ]:
resultados = pruebas.ejecutar_pruebas()

resumen = pruebas.resumen_pruebas(resultados)
print()
print('RESUMEN: ' + str(resumen['aprobadas']) + ' / ' + str(resumen['total'])
      + ' pruebas, ' + str(resumen['total_comprobaciones']) + ' comprobaciones, '
      + str(resumen['comprobaciones_fallidas']) + ' fallidas')

## 13. Conclusiones, supuestos y limitaciones

### Lo que se demostro

1. Las nueve formulas del documento funcionan tal cual, en Python, y coinciden con la hoja de
   control dentro de 0.000001.
2. Los cuatro escenarios salen de la misma funcion, y cada cambio queda trazado.
3. La sensibilidad la mide el motor, y el orden critico es un numero reproducible.
4. La IA recibe resultados ya calculados, y todo dato que menciona se valida contra esos
   resultados antes de mostrarse.
5. Sin clave de IA, el sistema sigue funcionando y **declara** que el texto lo genero el motor.
6. La pagina web no calcula nada: por lo tanto no puede discrepar del motor.

### Supuestos del modelo

- Un unico producto y una sola operacion. Sin mezcla ni multiples operaciones.
- El volumen planificado es un dato, no una prediccion.
- Los costos fijos del periodo se prorratean entre todas las unidades producidas.
- El precio de venta es el mismo para todas las unidades.
- El consumo de material es lineal con el volumen.
- No hay inventario ni tiempos de entrega.

### Limitaciones honestas

- El analisis es de corto plazo: no modela el valor del dinero en el tiempo.
- La sensibilidad es de un solo factor. Cambiar dos variables a la vez puede tener efectos que
  aqui no se ven.
- Los deltas optimista y adverso son parametros declarados, no probabilidades. No hay que leerlos
  como un pronostico.
- La calidad del texto de la IA depende de la version del modelo que este disponible.

### Reglas para trabajar con IA que este proyecto respeta

- La IA tiene una funcion concreta: interpretar sensibilidad y proponer alternativas medibles.
- Los calculos centrales son reproducibles con formulas, codigo y la hoja de control.
- Todo dato generado por IA se valida antes de utilizarse, y la validacion se muestra.
- No se usan datos personales ni informacion real de ninguna empresa.
- Un resultado de IA no se acepta si el equipo no puede explicar como se obtuvo.

La plantilla para llenar a mano durante la demo esta en `docs/bitacora_ia.md`.